In [14]:
import torch, time, os
import torch.nn as nn
from sklearn.metrics import classification_report


# ── same model definition as on host ────────────────────────────────────
class CNN2D(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x)
        return self.fc(x.view(x.size(0), -1))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [15]:
# match [B,27,9]
dummy_gpu = torch.randn(1, 27, 9).to(device)
dummy_cpu = dummy_gpu.cpu()

In [16]:
def benchmark(model, inp, device, iters=200):
    model.to(device).eval()
    # warm-up
    with torch.no_grad():
        for _ in range(20):
            _ = model(inp)
    # timed run
    torch.cuda.reset_peak_memory_stats() if device.type == "cuda" else None
    start = time.time()
    with torch.no_grad():
        for _ in range(iters):
            _ = model(inp)
    elapsed = time.time() - start
    mem = (torch.cuda.max_memory_allocated() / 1e6) if device.type == "cuda" else None
    return elapsed / iters * 1000, mem  # ms, MB or None

In [17]:
# Baseline
m0 = CNN2D().to(device)
m0.load_state_dict(torch.load("./artifacts/cnn2d.pth", map_location=device))
t0, mem0 = benchmark(m0, dummy_gpu, device)
print(f"Baseline    → {t0:.2f} ms, GPU mem peak={mem0:.1f} MB")

# Pruned
m1 = CNN2D().to(device)
m1.load_state_dict(torch.load("./artifacts/cnn2d_pruned.pth", map_location=device))
t1, mem1 = benchmark(m1, dummy_gpu, device)
print(f"Pruned 30%  → {t1:.2f} ms, GPU mem peak={mem1:.1f} MB")

Baseline    → 0.30 ms, GPU mem peak=9.6 MB
Pruned 30%  → 0.28 ms, GPU mem peak=9.6 MB


In [ ]:
# Dynamic-quantized (CPU only)
m2 = CNN2D().cpu().eval()
m2 = torch.quantization.quantize_dynamic(m2, {nn.Linear}, dtype=torch.qint8)
m2.load_state_dict(
    torch.load("./artifacts/cnn2d_quant_dynamic.pth", map_location="cpu")
)
td, _ = benchmark(m2, dummy_cpu, torch.device("cpu"))
print(
    f"Quant-dyn CPU → {td:.2f} ms, model file={os.path.getsize('./artifacts/cnn2d_quant_dynamic.pth')/1e6:.2f} MB"
)

# # Static-quantized ScriptModule (CPU)
# m3 = torch.jit.load("./artifacts/cnn2d_quant_static.pt", map_location="cpu")
# ts, _ = benchmark(m3, dummy_cpu, torch.device("cpu"))
# print(
#     f"Quant-stat CPU→ {ts:.2f} ms, file={os.path.getsize('./artifacts/cnn2d_quant_static.pt')/1e6:.2f} MB"
# )

Quant-dyn CPU → 0.31 ms, model file=0.02 MB
